In [1]:
# Math
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

# Predictive Modeling of Electric Vehicle Battery Degradation

## 1. Problem Formulation & Significance
As global electric vehicle (EV) adoption accelerates, accurately forecasting the Remaining Useful Life (RUL) and State of Health (SoH) of lithium-ion batteries is critical for vehicle valuation, warranty provisioning, and predictive maintenance. 

The two dominant battery chemistries: Nickel Manganese Cobalt (NMC) and Lithium Iron Phosphate (LFP), exhibit distinct degradation profiles under varying thermal and operational stresses. 

**Objective:**
This project transitions from exploratory data science into predictive machine learning. We aim to construct and evaluate multivariable models capable of predicting battery degradation. Because degradation modeling is highly susceptible to "feature dominance" (where algorithms rely entirely on obvious symptoms rather than underlying causes), we will conduct a three-part experiment:
1. **Baseline Modeling:** Predicting absolute SoH using all available telemetry and diagnostic data.
2. **Feature Ablation:** Removing direct diagnostic symptoms to force models to evaluate historical usage.
3. **Target Transformation:** Shifting the mathematical target from absolute capacity to a cycle-weighted degradation rate to uncover complex multivariable dependencies.

### 1.1 Dataset Assumptions & Constraints
**Disclaimer:** The dataset utilized in this project (`ev_battery_degradation_v1.csv`) is a synthetic dataset generated using a semi-empirical degradation model. 
* While it accurately reflects real-world physics, synthetic generation algorithms inherently hardcode strong linear correlations between primary variables (like Total Cycles) and the target variable (SoH). 
* A key focus of our machine learning methodology will be mitigating this synthetic linearity to prove our models can identify secondary, non-linear operational stresses (like fast-charging ratios and driving styles).

### 1.2 Mathematical Definition of the Baseline Target
In our initial phase, the target variable is the absolute State of Health ($SoH$), expressed as a percentage of the battery's original factory capacity. 

Let $C_{current}$ be the currently measurable capacity in kWh, and $C_{nominal}$ be the original capacity. 

$$SoH = \left( \frac{C_{current}}{C_{nominal}} \right) \times 100$$

In [2]:
data_path = 'data/ev_battery_degradation_v1.csv'
battery_data = pd.read_csv(data_path)

In [3]:
battery_data.shape

(10000, 13)

In [4]:
battery_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Vehicle_ID               10000 non-null  object 
 1   Car_Model                10000 non-null  object 
 2   Battery_Type             10000 non-null  object 
 3   Battery_Capacity_kWh     10000 non-null  float64
 4   Vehicle_Age_Months       10000 non-null  int64  
 5   Total_Charging_Cycles    10000 non-null  int64  
 6   Avg_Temperature_C        10000 non-null  float64
 7   Fast_Charge_Ratio        10000 non-null  float64
 8   Avg_Discharge_Rate_C     10000 non-null  float64
 9   Driving_Style            10000 non-null  object 
 10  Internal_Resistance_Ohm  10000 non-null  float64
 11  SoH_Percent              10000 non-null  float64
 12  Battery_Status           10000 non-null  object 
dtypes: float64(6), int64(2), object(5)
memory usage: 1015.8+ KB


## 2. Data Preparation & Feature Engineering

To prepare our dataset for machine learning algorithms, we must translate all categorical string data into mathematical representations. Before deciding on an encoding strategy, we first need to isolate our non-numeric columns and inspect their unique values and cardinality.

In [5]:
# Isolate columns with 'object' (string) data types
categorical_cols = battery_data.select_dtypes(include=['object']).columns

print("Categorical Columns and their Unique Values:\n" + "-"*45)

for col in categorical_cols:
    unique_vals = battery_data[col].unique()
    print(f"{col} ({len(unique_vals)} unique categories):")
    print(f"  {unique_vals}\n")

Categorical Columns and their Unique Values:
---------------------------------------------
Vehicle_ID (10000 unique categories):
  ['1fb46ae8' 'b7ef35aa' '76cb49e0' ... '3c549739' 'a6c88d13' '9e50c2e0']

Car_Model (5 unique categories):
  ['Tesla Model 3' 'Ford Mustang Mach-E' 'Hyundai Ioniq 5' 'Wuling Air EV'
 'BYD Atto 3']

Battery_Type (2 unique categories):
  ['NMC' 'LFP']

Driving_Style (3 unique categories):
  ['Aggressive' 'Conservative' 'Moderate']

Battery_Status (2 unique categories):
  ['Healthy' 'Replace Required']



Based on the programmatic inspection above, the encoding strategy depends heavily on the nature and cardinality of the variable to prevent introducing false mathematical relationships into our linear models.

* **Identifier Removal:** The `Vehicle_ID` column is completely unique to each row and will be dropped to prevent models from memorizing specific instances instead of learning generalized patterns.
* **Binary Encoding:** `Battery_Type` (NMC/LFP) and `Battery_Status` (Healthy/Replace Required) are binary. We will utilize `scikit-learn`'s `LabelEncoder` to map these to 0 and 1.
* **Ordinal Encoding:** `Driving_Style` has a strict hierarchical order (Conservative < Moderate < Aggressive). We will use a custom dictionary mapping to preserve this magnitude.
* **Nominal Encoding (Low Cardinality):** `Car_Model` contains 5 discrete categories with no numerical relationship. We will apply One-Hot Encoding (using `drop_first=True` to avoid the dummy variable trap) to prevent linear algorithms from assuming an ordinal hierarchy. 

In [6]:
# 1. Drop the unique identifier
battery_data = battery_data.drop(columns=['Vehicle_ID'])
print("Dropped Vehicle_ID column.")

# 2. Initialize LabelEncoders for binary columns
le_battery = LabelEncoder()
le_status = LabelEncoder()

# Apply Label Encoding
battery_data['Battery_Type_Encoded'] = le_battery.fit_transform(battery_data['Battery_Type'])
battery_data['Battery_Status_Encoded'] = le_status.fit_transform(battery_data['Battery_Status'])

# Print the mapping to prove we know what the model is doing
print("\nLabel Encoder Mappings:")
print(f"Battery_Type: {dict(zip(le_battery.classes_, le_battery.transform(le_battery.classes_)))}")
print(f"Battery_Status: {dict(zip(le_status.classes_, le_status.transform(le_status.classes_)))}")

# Drop the original text columns to keep the dataframe clean
battery_data = battery_data.drop(columns=['Battery_Type', 'Battery_Status'])

Dropped Vehicle_ID column.

Label Encoder Mappings:
Battery_Type: {'LFP': np.int64(0), 'NMC': np.int64(1)}
Battery_Status: {'Healthy': np.int64(0), 'Replace Required': np.int64(1)}


In [7]:
# 3. Ordinal Encoding for Driving Style
driving_mapping = {'Conservative': 0, 'Moderate': 1, 'Aggressive': 2}
battery_data['Driving_Style_Encoded'] = battery_data['Driving_Style'].map(driving_mapping)
battery_data = battery_data.drop(columns=['Driving_Style'])
print("Applied ordinal mapping to Driving_Style.")

# 4. One-Hot Encoding for Car Model
# We use drop_first=True to avoid the "dummy variable trap" (perfect multicollinearity)
battery_data = pd.get_dummies(battery_data, columns=['Car_Model'], drop_first=True, dtype=int)
print(f"Applied One-Hot Encoding to Car_Model.")

# Check the new shape and columns
print(f"\nNew dataset shape: {battery_data.shape}")
display(battery_data.head(10))

Applied ordinal mapping to Driving_Style.
Applied One-Hot Encoding to Car_Model.

New dataset shape: (10000, 15)


,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Internal_Resistance_Ohm,SoH_Percent,Battery_Type_Encoded,Battery_Status_Encoded,Driving_Style_Encoded,Car_Model_Ford Mustang Mach-E,Car_Model_Hyundai Ioniq 5,Car_Model_Tesla Model 3,Car_Model_Wuling Air EV
0,75.0,41,390,21.5,0.51,2.22,0.0362,94.60,1,0,2,0,0,1,0
1,75.0,29,401,18.0,0.62,1.34,0.0333,95.68,1,0,2,0,0,1,0
2,88.0,71,941,18.4,0.78,1.48,0.0526,89.80,1,0,0,1,0,0,0
3,88.0,57,378,10.8,0.61,0.72,0.0314,96.29,1,0,1,1,0,0,0
4,75.0,58,239,30.3,0.89,1.48,0.0297,96.75,1,0,0,0,0,1,0
5,72.6,27,225,20.0,0.35,2.50,0.0265,98.07,1,0,1,0,1,0,0
6,72.6,49,734,3.2,0.51,0.65,0.0401,94.20,1,0,0,0,1,0,0
7,72.6,12,121,27.4,0.29,1.41,0.0223,98.76,1,0,1,0,1,0,0
8,88.0,40,170,23.7,0.71,1.23,0.0236,98.53,1,0,0,1,0,0,0
9,26.0,12,100,19.6,0.59,1.50,0.0208,100.00,0,0,0,0,0,0,1
